# 01 — Exploratory Data Analysis

**Kaggle:** Add Input → *140k Real and Fake Faces*; Accelerator = None (CPU is enough); Internet = On.

Theory and explanation: [`docs/02_exploratory_data_analysis.md`](../docs/02_exploratory_data_analysis.md)

In [ ]:
# --- environment setup: works on Kaggle, Colab and locally ---
import sys, os, glob
REPO = "https://github.com/Mohd2Shaan/Deepfake-Detection-Project.git"
if os.path.exists("/kaggle/input"):                      # ---- Kaggle
    if not os.path.exists("/kaggle/working/project"):
        !git clone -q {REPO} /kaggle/working/project
    %cd /kaggle/working/project
    !pip install -q grad-cam
    # use the dataset attached via "Add Input" directly (read-only, nothing to download)
    for depth in range(1, 5):
        hits = [os.path.dirname(t) for t in glob.glob("/kaggle/input/" + "*/" * depth + "train")
                if os.path.isdir(os.path.join(os.path.dirname(t), "valid"))]
        if hits:
            os.environ["DEEPFAKE_DATA_DIR"] = hits[0]
            break
    else:
        print("WARNING: dataset not found - attach '140k Real and Fake Faces' via Add Input")
    # notebooks 03/04: copy trained models from the attached output of notebook 02
    import shutil
    for depth in range(0, 4):
        for ckpt in glob.glob("/kaggle/input/*/" + "*/" * depth + "checkpoints/*_best.pth"):
            if not os.path.exists(os.path.join("checkpoints", os.path.basename(ckpt))):
                shutil.copy(ckpt, "checkpoints/")
                print("using trained model", ckpt)
elif "google.colab" in sys.modules:                       # ---- Colab
    if not os.path.exists("/content/project"):
        !git clone -q {REPO} /content/project
    %cd /content/project
    !pip install -q -r requirements.txt kagglehub
elif os.path.basename(os.getcwd()) == "notebooks":        # ---- local
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

In [ ]:
# --- dataset (Colab/local: downloads ~4 GB via kagglehub the first time; Kaggle: already attached) ---
from src import config
if not (config.DATA_DIR / "train").exists():
    !python scripts/download_dataset.py --move
print("data dir:", config.DATA_DIR)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from src import eda, config
from src.dataset import get_dataloaders, save_batch_grid

## 1. Scan every image
The full dataset takes ~10 min on Kaggle. Set `MAX_PER_CLASS = 2000` for a quick look.

In [ ]:
MAX_PER_CLASS = None
df = eda.scan_images(eda.list_images(max_per_class=MAX_PER_CLASS))
df.head()

## 2. Class balance

In [ ]:
eda.plot_class_balance(df, config.PLOTS_DIR / 'eda_class_balance.png')

## 3. Image size, colour mode, corrupted files, duplicates

In [ ]:
print(df.groupby(['width', 'height']).size())
print(df['mode'].value_counts())
print('corrupted:', int(df['corrupted'].sum()))
dups = eda.find_duplicates(df)
print('duplicates:', len(dups))
dups.head()

## 4. Random samples

In [ ]:
for c in config.CLASS_NAMES:
    eda.plot_sample_grid(df, c, out_path=config.PLOTS_DIR / f'eda_samples_{c}.png')
    plt.show()

## 5. File size distribution (shortcut check)

In [ ]:
eda.plot_file_size_distribution(df, config.PLOTS_DIR / 'eda_file_size.png')
df.groupby('class')['file_kb'].describe()

## 6. Mean face and mean Fourier spectrum

In [ ]:
eda.plot_mean_faces_and_spectra(df, n=500, out_path=config.PLOTS_DIR / 'eda_mean_face_spectrum.png')

## 7. Sanity check — a batch after transforms

In [ ]:
loaders = get_dataloaders(num_workers=0)
for split, loader in loaders.items():
    print(split, len(loader.dataset), loader.dataset.class_counts())
save_batch_grid(loaders['train'], config.PLOTS_DIR / 'sample_train_batch.png')
from IPython.display import Image as Show
Show(str(config.PLOTS_DIR / 'sample_train_batch.png'))

## Observations

_Write findings here (class balance, sizes, corrupted/duplicates, any visible differences)._